<a href="https://colab.research.google.com/github/AzadMahmud/dl-lab/blob/main/assignment-1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from pathlib import Path
import numpy as np
import os
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score
import umap
import tensorflow as tf

In [ ]:
IMAGES_ROOT = Path(os.environ.get("IMAGES_ROOT", "/content/drive/MyDrive/images"))
OUT = Path("/content/outputs")
OUT.mkdir(parents=True, exist_ok=True)
(OUT / "predictions").mkdir(exist_ok=True)
(OUT / "figures").mkdir(exist_ok=True)

In [ ]:
def parse_label(name: str) -> str:
    if name.startswith("self_male_"):
        return "male_self"
    if name.startswith("second_male_"):
        return "male_second"
    if name.startswith("third_male_"):
        return "male_third"
    if name.startswith("first_flower_"):
        return "flower_first"
    if name.startswith("second_flower_"):
        return "flower_second"
    if name.startswith("third_flower_"):
        return "flower_third"
    if name.startswith("fourth_flower_"):
        return "flower_fourth"
    if name.startswith("fifth_flower_"):
        return "flower_fifth"
    return "unknown"


In [ ]:
def load_metadata(root: Path) -> pd.DataFrame:
    rows = []
    exts = ["*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"]
    for domain in ["male", "flower"]:
        files = []
        for ext in exts:
            files.extend(sorted((root / domain).glob(ext)))
        for p in files:
            rows.append(
                {
                    "image": p.name,
                    "path": str(p),
                    "domain": domain,
                    "label": parse_label(p.name),
                }
            )

    df = pd.DataFrame(rows, columns=["image", "path", "domain", "label"])
    if df.empty:
        raise FileNotFoundError(
            f"No images found in {root}. Expected folders: {root}/male and {root}/flower"
        )
    df.to_csv(OUT / "metadata.csv", index=False)
    return df


In [ ]:
MODELS = [
    ("VGG16", tf.keras.applications.VGG16, tf.keras.applications.vgg16.preprocess_input, tf.keras.applications.vgg16.decode_predictions, (224, 224)),
    ("VGG19", tf.keras.applications.VGG19, tf.keras.applications.vgg19.preprocess_input, tf.keras.applications.vgg19.decode_predictions, (224, 224)),
    ("ResNet50", tf.keras.applications.ResNet50, tf.keras.applications.resnet.preprocess_input, tf.keras.applications.resnet.decode_predictions, (224, 224)),
    ("ResNet101", tf.keras.applications.ResNet101, tf.keras.applications.resnet.preprocess_input, tf.keras.applications.resnet.decode_predictions, (224, 224)),
    ("DenseNet121", tf.keras.applications.DenseNet121, tf.keras.applications.densenet.preprocess_input, tf.keras.applications.densenet.decode_predictions, (224, 224)),
    ("MobileNetV2", tf.keras.applications.MobileNetV2, tf.keras.applications.mobilenet_v2.preprocess_input, tf.keras.applications.mobilenet_v2.decode_predictions, (224, 224)),
    ("InceptionV3", tf.keras.applications.InceptionV3, tf.keras.applications.inception_v3.preprocess_input, tf.keras.applications.inception_v3.decode_predictions, (299, 299)),
    ("Xception", tf.keras.applications.Xception, tf.keras.applications.xception.preprocess_input, tf.keras.applications.xception.decode_predictions, (299, 299)),
    ("EfficientNetB0", tf.keras.applications.EfficientNetB0, tf.keras.applications.efficientnet.preprocess_input, tf.keras.applications.efficientnet.decode_predictions, (224, 224)),
    ("NASNetMobile", tf.keras.applications.NASNetMobile, tf.keras.applications.nasnet.preprocess_input, tf.keras.applications.nasnet.decode_predictions, (224, 224)),
]

In [ ]:
def batch_preprocess(paths, size, preprocess, bs=8):
    for i in range(0, len(paths), bs):
        part = paths[i:i + bs]
        arr = []
        for p in part:
            img = tf.keras.utils.load_img(p, target_size=size)
            arr.append(tf.keras.utils.img_to_array(img))
        x = preprocess(np.array(arr, dtype=np.float32))
        yield i, part, x


In [ ]:
def embed_2d(x, method):
    if method == "PCA":
        return PCA(n_components=2, random_state=42).fit_transform(x)
    if method == "TSNE":
        return TSNE(n_components=2, random_state=42, init="pca", learning_rate="auto", perplexity=8).fit_transform(x)
    return umap.UMAP(n_components=2, random_state=42).fit_transform(x)


In [ ]:
def plot_2d(z, labels, title, save_path):
    labels = np.array(labels)
    classes = sorted(set(labels.tolist()))
    plt.figure(figsize=(7, 5))
    for i, c in enumerate(classes):
        idx = labels == c
        plt.scatter(z[idx, 0], z[idx, 1], s=40, alpha=0.9, label=c)
    plt.title(title)
    plt.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(save_path, dpi=250)
    plt.close()


In [ ]:
def run():
    df = load_metadata(IMAGES_ROOT)
    paths = df["path"].tolist()
    scores = []
    all_preds = []

    for name, ctor, pre, decode, size in MODELS:
        print("Running:", name)

        cls_model = ctor(weights="imagenet", include_top=True)
        rows = []
        for start, part, x in batch_preprocess(paths, size, pre):
            pred = cls_model.predict(x, verbose=0)
            topk = decode(pred, top=5)
            for j, k in enumerate(topk):
                top5 = " | ".join([f"{t[1]}:{t[2]:.4f}" for t in k])
                rows.append({
                    "model": name,
                    "image": df.iloc[start + j]["image"],
                    "domain": df.iloc[start + j]["domain"],
                    "label": df.iloc[start + j]["label"],
                    "top1": k[0][1],
                    "top1_prob": float(k[0][2]),
                    "top5": top5,
                })
        pred_df = pd.DataFrame(rows)
        pred_df.to_csv(OUT / "predictions" / f"pred_{name}.csv", index=False)
        all_preds.append(pred_df)
        tf.keras.backend.clear_session()

        feat_model = ctor(weights="imagenet", include_top=False, pooling="avg")
        feat_parts = []
        for _, _, x in batch_preprocess(paths, size, pre):
            feat_parts.append(feat_model.predict(x, verbose=0))
        feats = np.vstack(feat_parts)
        tf.keras.backend.clear_session()

        for domain in ["male", "flower"]:
            m = df["domain"] == domain
            x = feats[m.values]
            y = df.loc[m, "label"].tolist()
            y_int = pd.factorize(np.array(y))[0]

            for method in ["PCA", "TSNE", "UMAP"]:
                z = embed_2d(x, method)
                s = float(silhouette_score(z, y_int))
                scores.append({"model": name, "domain": domain, "method": method, "silhouette": s})
                plot_2d(z, y, f"{name} | {domain} | {method}", OUT / "figures" / f"{name}_{domain}_{method}.png")

    pd.concat(all_preds, ignore_index=True).to_csv(OUT / "all_predictions.csv", index=False)
    score_df = pd.DataFrame(scores)
    score_df.to_csv(OUT / "silhouette_scores.csv", index=False)
    print("Done. Files saved in", OUT)


run()

Running: VGG16
553467096/553467096 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step
35363/35363 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/usr/local/lib/python3.12/dist-packages/umap/umap_.py:2462: UserWarning: n_neighbors is larger than the dataset size; truncating to X.shape[0] - 1
  warn(
/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Running: VGG19
574710816/574710816 ━━━━━━━━━━━━━━━━━━━━ 8s 0us/step
80134624/80134624 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/usr/local/lib/python3.12/dist-packages/umap/umap_.py:2462: UserWarning: n_neighbors is larger than the dataset size; truncating to X.shape[0] - 1
  warn(
/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Running: ResNet50
102967424/102967424 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


...

Running: EfficientNetB0
21834768/21834768 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/usr/local/lib/python3.12/dist-packages/umap/umap_.py:2462: UserWarning: n_neighbors is larger than the dataset size; truncating to X.shape[0] - 1
  warn(
/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Running: NASNetMobile
24227760/24227760 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
19993432/19993432 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/usr/local/lib/python3.12/dist-packages/umap/umap_.py:2462: UserWarning: n_neighbors is larger than the dataset size; truncating to X.shape[0] - 1
  warn(
/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Done. Files saved in /content/outputs
